# Embeddings and Neural Models

So far, you’ve learned three major approaches to NLP:

- Bag of Words: counts words
- TF-IDF: weighs words based on importance
- Transformers: understand context and meaning

Now we take the next step, using dense vector representations to build small neural models in PyTorch. Modern NLP relies on these big ideas:

**Embedding** - turn text into compact numeric vectors where similar meaning produces similar vectors. Transformers generate these embeddings internally

**Semantic Search** - use embeddings to find related text based on meaning rather than keyword overlap

**Neural Networks in PyTorch** - build lightweight custom models that use embeddings as input, giving us more control than using transformer pipelines alone

This week, you’ll learn what embeddings are, explore how to use them for semantic search, and then build a simple text classifier in PyTorch using embedding features. This will tie all previous topics together and shows how modern NLP systems work behind the scenes.

## Embeddings

Before now, our text features were based on counts (Bag of Words) or weights (TF-IDF). These approaches work, but they have one major limitation: they don’t understand meaning. Modern NLP solves this by using embeddings, which turn text into dense numeric vectors that capture semantic relationships. In Week 3, you already used transformers to classify text. Those same models also generate the embeddings we’ll work with here. Embeddings act as the bridge between raw text and neural models.

An embedding is a compact vector that represents the meaning of a word, sentence, or document. Because embeddings encode meaning, texts with similar intent end up with similar vectors. For example, the phrases “good movie” and “great film” will appear close together in embedding space, while “bad experience” will sit far away from both. A small slice of a real embedding might look like:

```
[0.12, -0.48, 0.77, 1.03, -0.11, 0.22, ...]
```

This vector is usually much longer (hundreds of numbers), and is dense rather than sparse (unlike BoW and TF-IDF).

### How Are Embeddings Learned?

Transformers learn embeddings by reading huge amounts of text and noticing patterns. During training, the model tries to predict missing or next words, and it adjusts its internal weights each time it makes a mistake. As it does this over millions of sentences, words and sentences that appear in similar contexts start to get similar vectors.

The model doesn’t know the meaning of a word directly, it learns it from how the word is used. So “cat” and “dog” end up with similar embeddings because they show up in similar kinds of sentences, while something like “astronaut” ends up far away. These learned vectors are what allow us to compare meaning, perform semantic search, and use embeddings as inputs to neural models.

In [6]:
from sentence_transformers import SentenceTransformer
import torch

# Load a sentence embedding model
model = SentenceTransformer("all-MiniLM-L6-v2")

def embed(text):
    return torch.tensor(model.encode(text))

### How Are Embeddings Compared?

Once text has been converted into embeddings, we need a way to measure how similar two vectors are. For embeddings, the most common choice is **cosine similarity**. Cosine similarity compares the **direction** of two vectors rather than their length. If two vectors point in the same direction, they represent similar meaning.

- Value close to **1** = very similar meaning  
- Value near **0** = unrelated  
- Value less than **0** = very different meaning  

In [3]:
from torch.nn.functional import cosine_similarity

e1 = embed(["I loved this movie!"])
e2 = embed(["This film was fantastic!"])
e3 = embed(["The food was terrible."])

print(cosine_similarity(e1, e2))  
print(cosine_similarity(e1, e3))  
print(cosine_similarity(e3, e1))  

tensor([0.7652])
tensor([0.1573])
tensor([0.1573])


## Semantic Search

Embeddings let us go beyond matching keywords and instead match meaning. This idea is called semantic search. Traditional search methods like TF-IDF look for documents that contain the same words as the query. That works, but only when the phrasing is similar. Semantic search improves on this by comparing embeddings instead of raw text. Because embeddings capture meaning, we can find related documents even when they use different words.

For example, the query *"story about friendship"* might match a document like *"two kids go on a summer adventure and become close friends"*, even though no exact words overlap. This happens because both sentences produce embeddings with similar semantic structure.

The basic steps of semantic search are:

- Embed all documents once
- Embed the user’s query
- Compute cosine similarity between the query and each document
- Return the top k most similar documents

In [4]:
from sentence_transformers import SentenceTransformer
from torch.nn.functional import cosine_similarity

# embedding models
model = SentenceTransformer("all-MiniLM-L6-v2")

# document collection
docs = [
    "A young boy befriends a lost alien.",
    "Two detectives solve a mysterious crime.",
    "A group of friends goes on a summer adventure.",
    "An astronaut becomes stranded on Mars.",
    "A giant robot is stuck in a city."
]

# embed all documents
doc_embeddings = embed(docs)

def search(query, k=2):
    # embed the query
    q_emb = embed(query)
    
    # compute similarity to every document
    sims = cosine_similarity(q_emb, doc_embeddings)
   
    # get top k most similar
    top = sims.topk(k)
    
    print("Query:", query)
    print("\nTop results:")
    for score, idx in zip(top.values, top.indices):
        print(f"{score:.3f} - {docs[idx]}")

# evaluate
search("buddies go on a journey in the hot months")

Query: buddies go on a journey in the hot months

Top results:
0.512 - A group of friends goes on a summer adventure.
0.098 - A young boy befriends a lost alien.


## Neural Models

Now that we can turn text into meaningful embeddings, we can use those embeddings as inputs to a neural network. Unlike the Hugging Face pipeline from Week 3, which handled everything for us, building a PyTorch model gives us direct control over how the model is designed, trained, and evaluated.

You might wonder why we would bother with this when transformers already perform so well. In many cases, we would simply use a transformer model. However, there are situations where a transformer pipeline doesn’t do exactly what we need, or where we want a lightweight model that is easier to customize. In those cases, it’s useful to build a neural model that is better suited to the task.

A neural model is simply a series of layers that transforms input vectors (embeddings) into predictions. Because embeddings already capture a lot of meaning, the neural model we build can be very small and still perform well. A basic classifier typically includes an input layer that accepts the embedding, one or two hidden layers to process it, and an output layer that predicts a label.

Like you learned in deep learning, PyTorch makes this straightforward. You define a model, pass embeddings through it, compute loss, and update weights during training. Using embeddings this way is very common in real NLP systems: transformers generate the representations, and lightweight neural models sit on top to perform specific tasks.

In [2]:
from sentence_transformers import SentenceTransformer
import torch
import torch.nn as nn
import torch.optim as optim

# embedding model
embed_model = SentenceTransformer("all-MiniLM-L6-v2")

def embed(text):
    return torch.tensor(embed_model.encode(text))

# define neural model
class TextClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_classes):
        super().__init__()
        self.hidden = nn.Linear(input_dim, hidden_dim)
        self.relu = nn.ReLU()
        
        self.output = nn.Linear(hidden_dim, num_classes)
        self.log_softmax = nn.LogSoftmax(dim=1)

    def forward(self, x):
        x = self.hidden(x)
        x = self.relu(x)
        
        x = self.output(x)
        return self.log_softmax(x)

# dataset
texts = [
    "I loved this movie, it was great!",
    "What an amazing film, I enjoyed it.",
    "This was awful, I hated it.",
    "Terrible movie, really boring.",
]
labels = torch.tensor([1, 1, 0, 0])  # 1 = positive, 0 = negative

N_EPOCHS = 50
L_RATE = 0.003
X = embed(texts)
input_dim = X.shape[1]
model = TextClassifier(input_dim=input_dim, hidden_dim=64, num_classes=2)
optimizer = optim.Adam(model.parameters(), lr=L_RATE)
loss_fn = nn.NLLLoss()

# train loop
for epoch in range(N_EPOCHS):
    model.train()
    optimizer.zero_grad()
    output = model(X)
    loss = loss_fn(output, labels)
    loss.backward()
    optimizer.step()

# evaluate
model.eval()
with torch.no_grad():
    test_x = embed(["I really liked this movie"])
    output = model(test_x)
    pred = output.argmax(dim=1).item()
    print("prediction:", pred)


prediction: 1


# Summary

This week, we focused on tying together everything from the previous weeks and introduced the core ideas behind modern neural NLP.

## Embeddings

- What embeddings are and how they represent meaning as dense numeric vectors
- Saw how transformers generate embeddings
- Compared embeddings to BoW and TF-IDF

## Semantic Search

- Used a pretrained transformer model to create sentence embeddings
- Measured similarity between sentences using cosine similarity
- Saw how embeddings allow meaning based search instead of keyword matching

## Neural Models in PyTorch

- Built a small neural network that takes embeddings as input
- Trained the model on a simple text classification task
- Observed how embeddings let us use very small models while still getting strong performance